# COMP3122 — Lab 08 — Logistic Regression and Metrics
### Week 10 · Chapter 8

**Applied Data Science**

**This lab is graded.** It covers Chapter 8 in the order the chapter covers it.

**How it works:** each task has a cell for your answer, followed by a cell that checks it. Run the check cell and it prints `PASS`, or shows you what it expected.

**Your numbers are not the book's.** This lab uses a different split (`test_size=0.25`, `random_state=3122`), a different one-feature model, and 1,000 water pumps you have not seen.

**Derive your answers; do not type them.** Several tasks check how you got there as well — Task 3.1 checks which tumours landed in each part and that every label stayed with its own tumour, 3.2 reads what your scaler learned, and 3.4 and 7.1 check your probabilities come from your own model.

**How it is marked:** your mark is the number of answer checks that print `PASS`, out of **52**. The 3 `GUARD` checks in Task 7.4 earn no marks; they only warn you if the shared data was overwritten. Your professor also reads your code: where a task asks you to calculate or fit something, a result typed in rather than computed does not earn its check. Your own choices (6.1, 7.3, the word in 5.2) and the hand count Task 4.1 allows are fine.

**Task 6.1 asks you to write down a prediction before you run anything.** Do that.

**Nothing in this lab needs the internet.**

Work in order. The setup cell builds `X`, `target` and `pumps` and defines the checkers, so run it first. **Do not reassign those names** — build your answers into the variables each task asks for. If a check will not pass after five minutes, ask.

---
## Setup — run this first

In [ ]:
# Run this cell once, before you start. It builds the data and defines the checkers.
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")   # explain: silences a harmless CPU-count warning on the course server
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_score, recall_score, f1_score, roc_auc_score)

# ---- the breast cancer data from Chapter 8: 569 tumours, the ten "mean" columns ----
# No download. scikit-learn ships it with 0 = malignant, 1 = benign (Task 1.1 fixes that).
cancer = load_breast_cancer(as_frame=True)
X = cancer.data[[c for c in cancer.data.columns if c.startswith("mean")]]
target = cancer.target     # the ORIGINAL codes: 0 = malignant, 1 = benign

# ---- 1,000 water pumps, for Parts 1 and 7: data you have not seen before -----------
# Generated from a fixed seed, so every student gets exactly the same 1,000 rows.
# A pump fails more often when it vibrates more and has gone longer without service.
_rng = np.random.default_rng(3122)
_temp = _rng.normal(70, 8, 1000).round(1)
_vib = _rng.gamma(2.0, 1.5, 1000).round(2)
_hours = _rng.integers(0, 5000, 1000)
_z = -7.2 + 0.06 * (_temp - 70) + 0.75 * _vib + 0.0004 * _hours
pumps = pd.DataFrame({"temperature_c": _temp, "vibration_mm_s": _vib,
                      "hours_since_service": _hours,
                      "failed": (_rng.random(1000) < 1 / (1 + np.exp(-_z))).astype(int)})

# Untouched copies, so the Task 7.3 guard can tell whether anything was reassigned.
X_AT_START, TARGET_AT_START, PUMPS_AT_START = X.copy(), target.copy(), pumps.copy()

# Reference answers for the route checks, built from the untouched data. The checks compare
# your objects with these, so a number typed in, a shuffled label or a skipped step shows up.
_Y_REF = (TARGET_AT_START == 0).astype(int).rename("malignant")
_XTR_REF, _XTE_REF, _YTR_REF, _YTE_REF = train_test_split(
    X_AT_START, _Y_REF, test_size=0.25, random_state=3122)
TEST_ROWS_31 = sorted(_XTE_REF.index)
_SCALER_REF = StandardScaler().fit(_XTR_REF)   # fitted on a DataFrame, so it knows its column names
_PF = ["temperature_c", "vibration_mm_s", "hours_since_service"]
_PTR_REF, _PTE_REF, _QTR_REF, _QTE_REF = train_test_split(
    PUMPS_AT_START[_PF], PUMPS_AT_START["failed"], test_size=0.3, random_state=3122,
    stratify=PUMPS_AT_START["failed"])


def check(label, got, want):
    """Compare your answer with the expected value. Prints PASS or NOT YET.

    `got` may be a value, or a zero-argument function producing one. The second
    form is used wherever the check has to reach into your answer first — read an
    attribute, round a number, index into an array. Before the answer is filled in
    that would raise, and a traceback where a hint should be is not helpful, so it
    is caught and reported as NOT YET instead.
    """
    if callable(got):
        try:
            got = got()
        except Exception as e:
            print(f"NOT YET  {label}")
            print(f"         your answer raised {type(e).__name__}: {e}")
            return

    def _fail(why=None):
        print(f"NOT YET  {label}")
        if why:
            print(f"         {why}")
        print(f"         you gave: {got!r}")
        print(f"         expected: {want!r}")

    PANDAS = (pd.Series, pd.DataFrame)
    if isinstance(want, PANDAS) or isinstance(got, PANDAS):
        if type(got) is not type(want):
            return _fail(f"wrong type — got {type(got).__name__}, expected {type(want).__name__}")
        if list(got.index) != list(want.index):
            return _fail("the row labels do not match")
        if isinstance(want, pd.DataFrame) and list(got.columns) != list(want.columns):
            return _fail("the column labels do not match")
        try:
            ok = bool(got.equals(want))
        except Exception:
            ok = False
    else:
        if isinstance(got, (bool, np.bool_)) and not isinstance(want, (bool, np.bool_)):
            return _fail("True/False is not a number here")
        g, w = np.asarray(got), np.asarray(want)
        try:
            if g.shape == w.shape and g.dtype.kind in "fc" and w.dtype.kind in "fc":
                ok = bool(np.allclose(g, w))   # decimals: allow a tiny rounding difference
            else:
                ok = bool(np.array_equal(g, w))
        except Exception:
            ok = False

    if ok:
        print(f"PASS     {label}")
    else:
        _fail()


def whole(v):
    """A count or a position must be a whole number. 4 and 4.0 count as 4; 1.9 stays 1.9
    and does not pass. (Turning 1.9 into 1 would let a wrong answer through.)"""
    if isinstance(v, (bool, np.bool_)):
        return v
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)) and float(v).is_integer():
        return int(v)
    return v


def _same_rows(got, want):
    """None if `got` holds exactly the rows of `want` (in any order, none twice); else a reason."""
    rows = list(got.index)
    if len(set(rows)) != len(rows):
        return "some rows appear more than once"
    if set(rows) != set(want.index):
        return "not the rows the required split picks"
    return None


def labels_match(got, want):
    """True if `got` is a Series holding the same 0/1 labels as `want`, each on its own row.
    Row ORDER does not matter (sorting is fine); which label sits on which row does.
    Any whole-number or decimal type is fine (int32, int64, Int64, float64 ...).
    True/False values, text, missing values and wrong labels are not. Otherwise returns a short
    reason, which the check prints."""
    if not isinstance(got, pd.Series):
        return f"not a Series (got {type(got).__name__})"
    why = _same_rows(got, want)
    if why:
        return why
    if pd.api.types.is_bool_dtype(got.dtype):
        return "True/False values: make them 1/0 with .astype(int)"
    if not pd.api.types.is_numeric_dtype(got.dtype):
        return f"the values are not numbers (dtype {got.dtype})"
    if got.isna().any():
        return "it has missing values"
    w = want.loc[got.index]
    bad = int((got.to_numpy(dtype=float) != w.to_numpy(dtype=float)).sum())
    return True if bad == 0 else f"{bad} of {len(w)} labels are not on their own rows"


def paired_labels(features, labels, want):
    """True if `labels` is in the SAME row order as `features` (scikit-learn pairs rows by
    position, not by index) and holds the right label on every row. Reordering both together
    is fine; reordering only one of them is not."""
    if not isinstance(features, (pd.DataFrame, pd.Series)) or not isinstance(labels, pd.Series):
        return labels_match(labels, want)
    if not features.index.equals(labels.index):
        return ("the labels are not in the same row order as the features: "
                "put each feature part and its labels in the same order (reorder both together)")
    return labels_match(labels, want)


def rows_match(got, want):
    """True if the DataFrame `got` holds exactly the rows and columns of `want`, with the same
    values. Row order and column order do not matter."""
    if not isinstance(got, pd.DataFrame):
        return f"not a DataFrame (got {type(got).__name__})"
    why = _same_rows(got, want)
    if why:
        return why
    if sorted(got.columns) != sorted(want.columns) or len(set(got.columns)) != len(got.columns):
        return "not the expected columns"
    w = want.loc[got.index, list(got.columns)]
    return bool(np.allclose(got.to_numpy(dtype=float), w.to_numpy(dtype=float)))


def values_match(got, want):
    """True if the array `got` has the shape and values of `want` (tiny rounding allowed)."""
    g = np.asarray(got, dtype=float)
    if g.shape != np.shape(want):
        return f"shape {g.shape}, expected {np.shape(want)}"
    return bool(np.allclose(g, want))


def _labels_for(part, ref_labels):
    """The reference labels for the rows of `part`, in `part`'s own row order."""
    why = _same_rows(part, ref_labels)
    if why:
        raise ValueError(why)
    return ref_labels.loc[part.index]


def _scaled_for(part, ref_rows, ref_scaler):
    """What a correctly fitted scaler gives for the rows and columns of `part`, in its own order."""
    if rows_match(part, ref_rows) is not True:
        raise ValueError("the unscaled part is not the rows the required split picks")
    cols = list(ref_scaler.feature_names_in_)
    full = pd.DataFrame(ref_scaler.transform(ref_rows[cols]), index=ref_rows.index, columns=cols)
    return full.loc[part.index, list(part.columns)].to_numpy()


def _learned_means(scaler, ref_rows, cols=None):
    """True if `scaler` learned the means of `ref_rows`, in the column order it was fitted with."""
    cols = list(getattr(scaler, "feature_names_in_", cols if cols is not None else ref_rows.columns))
    return values_match(scaler.mean_, ref_rows[cols].mean().to_numpy())


def _model_probs(model, scaler, part, ref_rows):
    """Column [1] of `model` on the scaled reference rows of `part`, in `part`'s row order."""
    why = _same_rows(part, ref_rows)
    if why:
        raise ValueError(why)
    names = getattr(scaler, "feature_names_in_", None)
    cols = list(names) if names is not None else list(part.columns)
    rows = ref_rows.loc[part.index, cols]
    return model.predict_proba(scaler.transform(rows if names is not None else rows.to_numpy()))[:, 1]


def raises(fn):
    """Run fn() and return the name of the exception it raised, or 'no error'.

    Task 4.3 asks you to find out what a wrong line actually does. Guessing
    is not the same as knowing, and this is how you check.
    """
    try:
        fn()
    except Exception as e:
        return type(e).__name__
    return "no error"


print("Ready. X:", X.shape, " target:", target.shape, " pumps:", pumps.shape)
import sklearn
print("scikit-learn", sklearn.__version__, " pandas", pd.__version__, " numpy", np.__version__)


---
## Part 1 — The 95% trap  ·  §8.1

Before any model: which class are you hunting, and how rare is it?

**Task 1.1.** scikit-learn codes the tumours **0 = malignant, 1 = benign** (the setup cell
keeps those original codes in `target`). That is backwards for this lab.

Build `y`: a Series that is **1 where the tumour is malignant** and 0 otherwise (the numbers
1 and 0, not True/False), **named `"malignant"`**. Then store the share of malignant tumours, rounded to 4 decimals, in `t11_share`.

In [ ]:
y = (target == 0).astype(int).rename("malignant")
t11_share = round(float(y.mean()), 4)

In [ ]:
check('1.1 y is 1 where malignant', lambda: labels_match(y, _Y_REF), True)
check('1.1 y is named "malignant"', lambda: y.name, 'malignant')
check('1.1 share malignant', lambda: round(float(t11_share), 4), 0.3726)

**Task 1.2.** The `pumps` table holds 1,000 water pumps. `failed` is 1 for a pump that broke down.

- `t12_failures` — how many pumps failed
- fit a `DummyClassifier(strategy="most_frequent")` on the three feature columns and `failed`,
  and store its accuracy on the same 1,000 pumps in `t12_acc`
- `t12_caught` — how many of the failed pumps the dummy predicts as failed

In [ ]:
t12_failures = int(pumps["failed"].sum())
_pump_cols = ["temperature_c", "vibration_mm_s", "hours_since_service"]
_dummy_12 = DummyClassifier(strategy="most_frequent").fit(pumps[_pump_cols], pumps["failed"])
t12_acc = round(float(_dummy_12.score(pumps[_pump_cols], pumps["failed"])), 4)
t12_caught = int(((_dummy_12.predict(pumps[_pump_cols]) == 1) & (pumps["failed"] == 1)).sum())

In [ ]:
check('1.2 failed pumps', lambda: whole(t12_failures), 68)
check('1.2 lazy accuracy', lambda: round(float(t12_acc), 4), 0.932)
check('1.2 failures caught', lambda: whole(t12_caught), 0)

---
## Part 2 — From a label to a probability  ·  §8.2

Logistic regression is **a line, then a squash**.

**Task 2.1.** Write a function `sigmoid(z)` that returns `1 / (1 + e^(-z))`. Use `np.exp`, so it
works on a single number **and** on a NumPy array.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
check('2.1 sigmoid(0)', lambda: float(sigmoid(0)), 0.5)
check('2.1 sigmoid(2)', lambda: round(float(sigmoid(2)), 4), 0.8808)
check('2.1 on an array', lambda: np.round(sigmoid(np.array([-4.0, 1.5, 4.0])), 4), np.array([0.018, 0.8176, 0.982]))

**Task 2.2.** Fit a logistic regression on **one** feature, `mean texture`, unscaled.

Split `X[["mean texture"]]` and `y` with `test_size=0.2` and `random_state=15`, fit a
`LogisticRegression()` on the training part, and store it in `t22_model`. Then store the
texture at which the model is **exactly undecided** (probability 0.5, which is where z = 0),
rounded to 2 decimals, in `t22_undecided`.

In [ ]:
X22_train, X22_test, y22_train, y22_test = train_test_split(
    X[["mean texture"]], y, test_size=0.2, random_state=15
)
t22_model = LogisticRegression().fit(X22_train, y22_train)
t22_undecided = round(float(-t22_model.intercept_[0] / t22_model.coef_[0, 0]), 2)

In [ ]:
check('2.2 coefficient', lambda: round(float(t22_model.coef_[0, 0]), 4), 0.2423)
check('2.2 intercept', lambda: round(float(t22_model.intercept_[0]), 4), -5.2455)
check('2.2 undecided texture', lambda: round(float(t22_undecided), 2), 21.65)

---
## Part 3 — The scikit-learn workflow  ·  §8.3

All ten mean features. **Not the chapter's split**: this lab uses `test_size=0.25` and
`random_state=3122`, so your numbers will differ from the book's.

**Task 3.1.** Split `X` and `y` into `X_train`, `X_test`, `y_train`, `y_test` with
`test_size=0.25` and `random_state=3122`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=3122)

In [ ]:
check('3.1 X_train and X_test are the rows random_state=3122 picks',
      lambda: (rows_match(X_train, _XTR_REF), rows_match(X_test, _XTE_REF)), (True, True))
check('3.1 each label stays with its own tumour, in step with the features',
      lambda: (paired_labels(X_train, y_train, _YTR_REF), paired_labels(X_test, y_test, _YTE_REF)), (True, True))
check('3.1 test malignant count', lambda: whole(int(y_test.sum())), 53)

**Task 3.2.** Create a `StandardScaler` called `scaler`, fit it on the **training** features only,
and use it to make `X_train_s` and `X_test_s`.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
check('3.2 scaler learned the TRAINING means, not all 569',
      lambda: _learned_means(scaler, _XTR_REF, list(X_train.columns)), True)
check('3.2 X_train_s and X_test_s are the scaled training and test rows',
      lambda: (values_match(X_train_s, _scaled_for(X_train, _XTR_REF, _SCALER_REF)),
               values_match(X_test_s, _scaled_for(X_test, _XTE_REF, _SCALER_REF))), (True, True))

**Task 3.3.** Fit a `LogisticRegression()` called `model` on the scaled training data, and store
its accuracy on the scaled **test** data in `t33_acc`.

In [ ]:
model = LogisticRegression().fit(X_train_s, y_train)
t33_acc = round(float(model.score(X_test_s, y_test)), 4)

In [ ]:
check('3.3 model is a fitted LogisticRegression on 10 features',
      lambda: isinstance(model, LogisticRegression) and whole(model.n_features_in_) == 10, True)
check('3.3 test accuracy (yours, and recomputed from model)', lambda: (round(float(t33_acc), 4),
      round(float(model.score(_scaled_for(X_test, _XTE_REF, _SCALER_REF), _labels_for(X_test, _YTE_REF))), 4)),
      (0.9301, 0.9301))

**Task 3.4.** Store the probability of **malignant** for every test tumour in `t34_p` (one
number per tumour). Check `model.classes_` first to be sure which column that is. Then store
how many test tumours have a probability **above 0.5** in `t34_flagged`.

In [ ]:
# model.classes_ is [0, 1], so column 1 is P(malignant)
t34_p = model.predict_proba(X_test_s)[:, 1]
t34_flagged = int((t34_p > 0.5).sum())

In [ ]:
check('3.4 column [1] of this model', lambda: bool(np.allclose(t34_p, model.predict_proba(X_test_s)[:, 1])), True)
check('3.4 one probability per test tumour', lambda: len(t34_p), 143)
check('3.4 tumours above 0.5', lambda: whole(t34_flagged), 49)

---
## Part 4 — The confusion matrix  ·  §8.4

Stop pooling. Four kinds of outcome, each counted separately.

**Task 4.1.** Twelve patients. `1` = has the disease, `0` = healthy.

```
truth = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0]
said  = [1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0]
```

Count by hand or with masks, and store `t41_tp`, `t41_fn`, `t41_fp`, `t41_tn` and the
accuracy `t41_acc` (rounded to 4 decimals).

In [ ]:
_truth_41 = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0])
_said_41 = np.array([1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0])
t41_tp = int(((_truth_41 == 1) & (_said_41 == 1)).sum())
t41_fn = int(((_truth_41 == 1) & (_said_41 == 0)).sum())
t41_fp = int(((_truth_41 == 0) & (_said_41 == 1)).sum())
t41_tn = int(((_truth_41 == 0) & (_said_41 == 0)).sum())
t41_acc = round(float((_truth_41 == _said_41).mean()), 4)

In [ ]:
check('4.1 TP', lambda: whole(t41_tp), 3)
check('4.1 FN', lambda: whole(t41_fn), 2)
check('4.1 FP', lambda: whole(t41_fp), 1)
check('4.1 TN', lambda: whole(t41_tn), 6)
check('4.1 accuracy', lambda: round(float(t41_acc), 4), 0.75)

**Task 4.2.** Store the confusion matrix of `model` on the test set in `t42_cm` (rows are the
truth, columns the prediction). Then read from it, into `t42_fn` and `t42_fp`, how many cancers
were **missed** and how many **false alarms** there were.

In [ ]:
t42_cm = confusion_matrix(y_test, model.predict(X_test_s))
t42_fn = int(t42_cm[1, 0])
t42_fp = int(t42_cm[0, 1])

In [ ]:
check('4.2 confusion matrix', lambda: np.asarray(t42_cm), np.array([[87, 3], [7, 46]]))
check('4.2 missed cancers (FN)', lambda: whole(t42_fn), 7)
check('4.2 false alarms (FP)', lambda: whole(t42_fp), 3)

**Task 4.3.** What happens if you pass the **probabilities** to `confusion_matrix` instead of
labels? Do not guess: use the `raises` helper from the setup cell and store its answer in
`t43_error`:

```python
t43_error = raises(lambda: confusion_matrix(y_test, t34_p))
```

In [ ]:
t43_error = raises(lambda: confusion_matrix(y_test, t34_p))

In [ ]:
check('4.3 the error it raises', t43_error, 'ValueError')

---
## Part 5 — Precision, recall and F1  ·  §8.5

**Task 5.1.** From the counts in `t42_cm`, compute by hand (no scikit-learn metric functions)
`t51_precision`, `t51_recall` and `t51_f1`, each rounded to 4 decimals.

In [ ]:
_tp_51 = int(t42_cm[1, 1])
_fp_51 = int(t42_cm[0, 1])
_fn_51 = int(t42_cm[1, 0])
_p_51 = _tp_51 / (_tp_51 + _fp_51)
_r_51 = _tp_51 / (_tp_51 + _fn_51)
t51_precision = round(_p_51, 4)
t51_recall = round(_r_51, 4)
t51_f1 = round(2 * _p_51 * _r_51 / (_p_51 + _r_51), 4)

In [ ]:
check('5.1 precision', lambda: round(float(t51_precision), 4), 0.9388)
check('5.1 recall', lambda: round(float(t51_recall), 4), 0.8679)
check('5.1 F1', lambda: round(float(t51_f1), 4), 0.902)

**Task 5.2.** The forgotten recoding. Split `X` and the **original** `target` exactly as in
Task 3.1, fit a `LogisticRegression()` on the scaled training data (use your `scaler`), and
store `precision_score` on the test set, rounded to 4 decimals, in `t52_precision`.

Then answer in `t52_about`: is that number about finding `'cancer'` or about finding `'healthy'`
patients?

In [ ]:
_, _, _y52_train, _y52_test = train_test_split(X, target, test_size=0.25, random_state=3122)
_m52 = LogisticRegression().fit(X_train_s, _y52_train)
t52_precision = round(float(precision_score(_y52_test, _m52.predict(X_test_s))), 4)
t52_about = 'healthy'

In [ ]:
check('5.2 precision on the original coding', lambda: round(float(t52_precision), 4), 0.9255)
check('5.2 what that number is about', t52_about, 'healthy')

---
## Part 6 — Moving the threshold  ·  §8.6

**Task 6.1 – predict first.** You will lower the threshold from 0.5 to 0.3, without retraining.
Before you measure anything: which statement about the number of **missed cancers** is
**guaranteed**, for any data? Store `'down or same'` or `'up or same'` in `t61_fn_goes`.

In [ ]:
# Prediction before running: lowering the threshold flags at least as many positives, so FN goes down or stays the same.
t61_fn_goes = 'down or same'

In [ ]:
check('6.1 missed cancers when the threshold drops', t61_fn_goes, 'down or same')

**Task 6.2.** Label the test tumours with your own threshold of **0.3** using `t34_p`. Store the
confusion matrix in `t62_cm`, and the recall and precision (rounded to 4 decimals) in
`t62_recall` and `t62_precision`.

In [ ]:
_y62_pred = (t34_p >= 0.3).astype(int)
t62_cm = confusion_matrix(y_test, _y62_pred)
t62_recall = round(float(recall_score(y_test, _y62_pred)), 4)
t62_precision = round(float(precision_score(y_test, _y62_pred)), 4)

In [ ]:
check('6.2 matrix at 0.3', lambda: np.asarray(t62_cm), np.array([[84, 6], [2, 51]]))
check('6.2 recall at 0.3', lambda: round(float(t62_recall), 4), 0.9623)
check('6.2 precision at 0.3', lambda: round(float(t62_precision), 4), 0.8947)

**Task 6.3.** Store the AUC computed from the **probabilities** in `t63_auc`, and the AUC computed
(wrongly) from the 0/1 labels `model.predict(X_test_s)` in `t63_auc_labels`. Round both to 4
decimals.

In [ ]:
t63_auc = round(float(roc_auc_score(y_test, t34_p)), 4)
t63_auc_labels = round(float(roc_auc_score(y_test, model.predict(X_test_s))), 4)

In [ ]:
check('6.3 AUC from probabilities', lambda: round(float(t63_auc), 4), 0.9899)
check('6.3 AUC from labels', lambda: round(float(t63_auc_labels), 4), 0.9173)

---
## Part 7 — The metric that matches the cost  ·  §8.7

Back to the pumps. A **missed failure** stops a town's water supply; a **needless inspection**
costs an hour of an engineer's time.

**Task 7.1.** Split the three pump features and `failed` into `P_train`, `P_test`, `q_train`,
`q_test` with `test_size=0.3`, `random_state=3122` and **stratify=pumps["failed"]** (so the
test set keeps the 6.8% rate). Fit a `StandardScaler` called `pump_scaler` on the training rows
only, fit a `LogisticRegression()` called `pump_model` on the scaled training rows, and store
the test-set probabilities of failure in `t71_p`. Then store the recall at the default 0.5 threshold,
rounded to 4 decimals, in `t71_recall`.

In [ ]:
_pump_features = ["temperature_c", "vibration_mm_s", "hours_since_service"]
P_train, P_test, q_train, q_test = train_test_split(
    pumps[_pump_features], pumps["failed"], test_size=0.3, random_state=3122, stratify=pumps["failed"]
)
pump_scaler = StandardScaler().fit(P_train)
pump_model = LogisticRegression().fit(pump_scaler.transform(P_train), q_train)
t71_p = pump_model.predict_proba(pump_scaler.transform(P_test))[:, 1]
t71_recall = round(float(recall_score(q_test, (t71_p > 0.5).astype(int))), 4)

In [ ]:
check('7.1 the stratified pump rows, each label in step with its row (train and test)',
        lambda: (rows_match(P_test, _PTE_REF) is True and paired_labels(P_train, q_train, _QTR_REF),
                 paired_labels(P_test, q_test, _QTE_REF)), (True, True))
check('7.1 pump_scaler learned the training pumps only',
        lambda: _learned_means(pump_scaler, _PTR_REF, list(P_train.columns)), True)
check('7.1 t71_p is pump_model\'s failure column on the scaled test pumps',
        lambda: isinstance(pump_model, LogisticRegression) and values_match(
            t71_p, _model_probs(pump_model, pump_scaler, P_test, _PTE_REF)), True)
check('7.1 recall at 0.5 (yours, and recomputed from t71_p)', lambda: (round(float(t71_recall), 4),
        round(float(recall_score(_labels_for(P_test, _QTE_REF), (np.asarray(t71_p) > 0.5).astype(int))), 4)),
        (0.35, 0.35))

**Task 7.2.** A missed failure costs **20**; a needless inspection costs **1**. For each threshold
in `[0.5, 0.4, 0.3, 0.2, 0.1, 0.05]`, compute `20 × FN + 1 × FP` on the test pumps. Store the six
totals, in that order, as a list in `t72_costs`, and the cheapest threshold in `t72_best`.

In [ ]:
_thresholds_72 = [0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
t72_costs = []
for _th in _thresholds_72:
    _tn_th, _fp_th, _fn_th, _tp_th = confusion_matrix(q_test, (t71_p > _th).astype(int)).ravel()
    t72_costs.append(int(20 * _fn_th + 1 * _fp_th))
t72_best = _thresholds_72[int(np.argmin(t72_costs))]

In [ ]:
check('7.2 the six costs', lambda: [whole(c) for c in t72_costs], [262, 243, 164, 107, 99, 109])
check('7.2 cheapest threshold', lambda: float(t72_best), 0.1)

**Task 7.3.** Each scenario ends with a **claim**. Answer `'trustworthy'` if the evidence
supports the claim, or `'not trustworthy'` if it does not:

- `t73_a` — the pump model scores 0.95 accuracy on the test pumps. Claim: it catches most failures.
- `t73_b` — the tumour model's AUC is 0.9899. Claim: at the 0.5 threshold it misses almost no cancers.
- `t73_c` — after recoding so 1 = malignant, the malignant row of `classification_report` shows
  recall 0.87. Claim: on this test set the model caught about 87% of the cancers.
- `t73_d` — precision 0.93, computed with scikit-learn's **original** coding. Claim: when the
  model says *cancer*, it is right 93% of the time.

In [ ]:
t73_a = 'not trustworthy'
t73_b = 'not trustworthy'
t73_c = 'trustworthy'
t73_d = 'not trustworthy'

In [ ]:
check('7.3 a — accuracy 0.95 on pumps', t73_a, 'not trustworthy')
check('7.3 b — high AUC at 0.5', t73_b, 'not trustworthy')
check('7.3 c — malignant-row recall', t73_c, 'trustworthy')
check('7.3 d — precision on the original coding', t73_d, 'not trustworthy')

**Task 7.4 — GUARD.** Nothing to write.

The check below reads `X`, `target` and `pumps` directly and confirms they still hold what the
setup cell put there. It passes on an untouched notebook, so it is not evidence of work – it
exists to fail loudly if something above overwrote the shared data.

In [ ]:
# Nothing to write for this task.

In [ ]:
check('7.4 GUARD X untouched', lambda: X.equals(X_AT_START), True)
check('7.4 GUARD target untouched', lambda: target.equals(TARGET_AT_START), True)
check('7.4 GUARD pumps untouched', lambda: pumps.equals(PUMPS_AT_START), True)

---
## Finish

Choose **Kernel → Restart Kernel and Run All Cells** one last time. If every check prints `PASS` from a clean start, your work is reproducible and you are done.

**Two answers to carry into the quiz:**

- Make the class you are hunting label 1 before you compute any metric (Tasks 1.1 and 5.2).
- The threshold is a choice. When you know what each error costs, choose it by cost (Tasks 6.2 and 7.2).

---
## Your submission line

Run the cell below and paste **the line it prints** into *Lab 8 — submit* on D2L. Nothing else – not the address bar of this notebook, which carries a port number that stops working when your server restarts. The line is built from your account name, so it always works.

In [1]:
# Run this cell. Copy the line it prints into 'Lab 8 -- submit' on D2L. Nothing else.
from pathlib import Path
account = Path.home().name               # your gblearn account name
print(f'http://{account}.gblearn.com/python/?open=comp3122/labs/lab08/COMP3122-Lab08-Student.ipynb')

http://f6172332.gblearn.com/python/?open=comp3122/labs/lab08/COMP3122-Lab08-Student.ipynb
